# CS5494 — MNIST generative models
**Zian LI — student ID 60444432 (EID zli934)**

上传本 notebook 到 Colab，选择 GPU 运行时，再按顺序运行。无需远程连接助手。
默认将 checkpoint 保存到 Google Drive，重开运行时后重复运行相同配置即可续跑。
正式实验分为“三个主模型”和“三项消融”，都完成后导出结果 ZIP。

This notebook embeds all local source files. It trains FVSBN, MADE and GAN,
records actual measurements, and exports evidence for the final English report.
Reduced smoke runs are clearly labelled and must not be used as final results.


## 1. Configuration and persistent output directory

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

USE_DRIVE = True  # Recommended on Colab: checkpoints survive runtime deletion.
SMOKE = False    # True runs 512 examples × 2 epochs for pipeline verification only.
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT = Path("/content/drive/MyDrive/CS5494_MNIST") / ("smoke_runs" if SMOKE else "runs")
else:
    OUTPUT = (Path("/content") if IN_COLAB else Path.cwd()) / "CS5494_MNIST" / ("smoke_runs" if SMOKE else "runs")
SOURCE = (Path("/content") if IN_COLAB else Path.cwd()) / "cs5494_mnist_source"
SOURCE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA = SOURCE / "data"
os.chdir(SOURCE)
print("Source:", SOURCE, "\nPersistent results:", OUTPUT)

# Keep Colab's CUDA-compatible torch; install only missing lightweight packages.
for package in ["numpy", "matplotlib"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name())
elif not SMOKE:
    raise RuntimeError("Formal configuration: select a GPU runtime, reconnect, and rerun. Set SMOKE=True only for a CPU check.")


## 2. Embedded source files
These cells write the implementation into the source directory. Reviewable copies accompany this notebook in the repository.

### models.py

In [ ]:
%%writefile models.py
"""Three required generative models; no torchvision dependency."""
import torch
from torch import nn
from torch.nn import functional as F


class MaskedLinear(nn.Linear):
    def __init__(self, in_features, out_features, mask):
        super().__init__(in_features, out_features)
        self.register_buffer("mask", torch.as_tensor(mask, dtype=torch.float32))

    def forward(self, x):
        return F.linear(x, self.weight * self.mask, self.bias)


class FVSBN(nn.Module):
    """p(x)=product_i Bernoulli(x_i; sigmoid(b_i + sum_{j<i} W_ij*x_j))."""
    def __init__(self, dimensions=784):
        super().__init__()
        self.dimensions = dimensions
        self.linear = MaskedLinear(dimensions, dimensions,
                                   torch.tril(torch.ones(dimensions, dimensions), diagonal=-1))

    def forward(self, x):
        return self.linear(x)

    @torch.no_grad()
    def sample(self, uniforms):
        # Incremental exact ancestral sampling avoids 784 full matrix multiplies.
        n, dimensions = uniforms.shape
        samples = torch.zeros_like(uniforms)
        logits = self.linear.bias.unsqueeze(0).expand(n, -1).clone()
        weights = self.linear.weight * self.linear.mask
        for i in range(dimensions):
            samples[:, i] = (uniforms[:, i] < logits[:, i].sigmoid()).to(samples.dtype)
            logits += samples[:, i:i+1] * weights[:, i].unsqueeze(0)
        return samples


class MADE(nn.Module):
    """Natural-order masked neural autoregressor, with fixed connectivity degrees.

    Hidden edges require m(previous)<=m(next); output edges require m(hidden)<i.
    Consequently output i depends only on pixels j<i. No input copying is possible.
    """
    def __init__(self, dimensions=784, hidden=512, depth=1, mask_seed=5494):
        super().__init__()
        self.dimensions = dimensions
        generator = torch.Generator().manual_seed(mask_seed)
        degrees = [torch.arange(1, dimensions + 1)]
        degrees += [torch.randint(1, dimensions, (hidden,), generator=generator) for _ in range(depth)]
        degrees += [torch.arange(1, dimensions + 1)]
        modules = []
        for i, (previous, following) in enumerate(zip(degrees, degrees[1:])):
            output = i == len(degrees) - 2
            mask = following[:, None] > previous[None, :] if output else following[:, None] >= previous[None, :]
            modules.append(MaskedLinear(len(previous), len(following), mask))
            if not output:
                modules.append(nn.ReLU())
        self.net = nn.Sequential(*modules)

    def forward(self, x):
        return self.net(x)

    @torch.no_grad()
    def sample(self, uniforms):
        samples = torch.zeros_like(uniforms)
        for i in range(self.dimensions):
            probability = self(samples)[:, i].sigmoid()
            samples[:, i] = (uniforms[:, i] < probability).to(samples.dtype)
        return samples


class Generator(nn.Module):
    def __init__(self, latent=100):
        super().__init__()
        self.latent = latent
        self.net = nn.Sequential(nn.Linear(latent, 256), nn.LeakyReLU(.2),
                                 nn.Linear(256, 512), nn.LeakyReLU(.2),
                                 nn.Linear(512, 784), nn.Tanh())

    def forward(self, noise):
        return self.net(noise)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, 512), nn.LeakyReLU(.2),
                                 nn.Linear(512, 256), nn.LeakyReLU(.2), nn.Linear(256, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)  # Logits, not sigmoid probabilities.


### data.py

In [ ]:
%%writefile data.py
"""Download and verify the original MNIST IDX files, then make a fixed split."""
import gzip
import hashlib
import json
from pathlib import Path
import struct
import urllib.request

import numpy as np
import torch

# Same public mirror and original checksums used by torchvision.datasets.MNIST.
MIRROR = "https://storage.googleapis.com/cvdf-datasets/mnist/"
FILES = {
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}


def load_data(directory, split_seed=5494):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    records = {}
    arrays = {}
    for name, checksum in FILES.items():
        assert len(checksum) == 32
        path = directory / name
        if not path.exists() or hashlib.md5(path.read_bytes()).hexdigest() != checksum:
            print("Downloading", name, flush=True)
            temporary = path.with_suffix(path.suffix + ".partial")
            with urllib.request.urlopen(MIRROR + name, timeout=90) as response:
                temporary.write_bytes(response.read())
            if hashlib.md5(temporary.read_bytes()).hexdigest() != checksum:
                raise ValueError("MNIST checksum mismatch: " + name)
            temporary.replace(path)
        content = gzip.decompress(path.read_bytes())
        magic, count = struct.unpack(">II", content[:8])
        if magic == 2051:
            height, width = struct.unpack(">II", content[8:16])
            assert (height, width) == (28, 28)
            arrays[name] = np.frombuffer(content, dtype=np.uint8, offset=16).copy().reshape(count, 784)
        elif magic == 2049:
            arrays[name] = np.frombuffer(content, dtype=np.uint8, offset=8).copy()
            assert len(arrays[name]) == count
        else:
            raise ValueError("Invalid IDX magic")
        records[name] = {"url": MIRROR + name, "md5": checksum,
                         "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    images = torch.from_numpy(arrays["train-images-idx3-ubyte.gz"])
    labels = torch.from_numpy(arrays["train-labels-idx1-ubyte.gz"])
    indices = np.random.default_rng(split_seed).permutation(60000)
    training, validation = indices[:50000], indices[50000:]
    (directory / "manifest.json").write_text(json.dumps({"files": records, "split_seed": split_seed,
                 "train_count": 50000, "validation_count": 10000, "test_count": 10000,
                 "training_index_sha256": hashlib.sha256(training.tobytes()).hexdigest()}, indent=2))
    np.savez_compressed(directory / "split_indices.npz", train=training, validation=validation)
    return {"train": images[training], "validation": images[validation],
            "test": torch.from_numpy(arrays["t10k-images-idx3-ubyte.gz"]),
            "train_labels": labels[training], "validation_labels": labels[validation],
            "test_labels": torch.from_numpy(arrays["t10k-labels-idx1-ubyte.gz"])}


def preprocess(images, binary):
    # 128/255 >= .5; original grayscale is retained for GAN inputs.
    return (images >= 128).float() if binary else images.float().div(127.5).sub(1)


### check_models.py

In [ ]:
%%writefile check_models.py
"""Causal, normalized-density, gradient and GAN-update checks on small models."""
import itertools
import json
from pathlib import Path

import torch
from torch.nn import functional as F

from models import FVSBN, MADE, Generator, Discriminator


def run_checks():
    torch.manual_seed(5494)
    checks = []
    for model in [FVSBN(6), MADE(6, hidden=12, depth=1), MADE(6, hidden=12, depth=2)]:
        model = model.double()
        x = torch.rand(6, dtype=torch.float64, requires_grad=True)
        jacobian = torch.autograd.functional.jacobian(lambda z: model(z[None])[0], x)
        assert torch.triu(jacobian).abs().max().item() == 0, "Current/future-pixel leakage"
        for i in range(6):
            changed = x.detach().clone()
            changed[i:] = 1 - changed[i:]
            torch.testing.assert_close(model(x[None])[:, :i+1], model(changed[None])[:, :i+1])
        configurations = torch.tensor(list(itertools.product([0., 1.], repeat=6)), dtype=torch.float64)
        logp = -F.binary_cross_entropy_with_logits(model(configurations), configurations, reduction="none").sum(1)
        torch.testing.assert_close(logp.exp().sum(), torch.tensor(1., dtype=torch.float64))
        # Finite difference through the network, independent of the mask proof.
        assert torch.autograd.gradcheck(model, (x[None],), eps=1e-6, atol=1e-4)
        model(configurations).sum().backward()
        for module in model.modules():
            if hasattr(module, "mask"):
                assert (module.weight.grad[module.mask == 0] == 0).all()
        random_values = torch.rand(12, 6, dtype=torch.float64)
        generated = model.sample(random_values)
        assert ((generated == 0) | (generated == 1)).all()
        if isinstance(model, FVSBN):
            reference = torch.zeros_like(random_values)
            for i in range(6):
                reference[:, i] = (random_values[:, i] < model(reference)[:, i].sigmoid()).double()
            torch.testing.assert_close(generated, reference)
        checks.append({"model": type(model).__name__, "layers": len(list(model.modules())),
                       "causality": True, "joint_probability_sum": logp.exp().sum().item(),
                       "finite_difference": True, "masked_gradients_zero": True})
    g, d = Generator(), Discriminator()
    noise = torch.randn(8, 100)
    fake = g(noise)
    assert fake.shape == (8, 784) and fake.min() >= -1 and fake.max() <= 1
    d_loss = F.binary_cross_entropy_with_logits(d(fake.detach()), torch.zeros(8))
    d_loss.backward()
    assert all(p.grad is None for p in g.parameters()), "D step must detach G output"
    d.zero_grad(set_to_none=True)
    for p in d.parameters():
        p.requires_grad_(False)
    g_loss = F.binary_cross_entropy_with_logits(d(g(noise)), torch.ones(8))
    g_loss.backward()
    assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in g.parameters())
    assert all(p.grad is None for p in d.parameters())
    return {"passed": True, "autoregressive_checks": checks, "gan_gradient_routing": True}


if __name__ == "__main__":
    result = run_checks()
    Path("model_checks.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))


### train.py

In [ ]:
%%writefile train.py
"""Train all required MNIST models and controlled ablations, with resumable runs.

Examples:
  python train.py --suite main --device cuda --output /content/drive/MyDrive/CS5494/runs
  python train.py --suite all --device cuda --output /content/drive/MyDrive/CS5494/runs
  python train.py --suite main --smoke --device cpu --output smoke_runs
Checkpoints are our own locally produced files; never load untrusted .pt files.
"""
import argparse
import copy
import hashlib
import json
import math
import platform
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

from data import load_data, preprocess
from models import Discriminator, FVSBN, Generator, MADE


@dataclass
class Experiment:
    name: str
    model: str
    epochs: int
    lr: float
    n_train: int = 50000
    batch_size: int = 256
    hidden: int = 512
    depth: int = 1
    seed: int = 5494


MAIN = [Experiment("fvsbn_main", "fvsbn", 30, .001),
        Experiment("made_main", "made", 40, .001),
        Experiment("gan_main", "gan", 50, .0002)]
ABLATIONS = [Experiment("made_depth2", "made", 40, .001, depth=2),
             Experiment("made_10k", "made", 40, .001, n_train=10000),
             Experiment("gan_lr1e4", "gan", 50, .0001)]


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def atomic_json(path, data):
    temporary = path.with_suffix(path.suffix + ".partial")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def atomic_checkpoint(path, data):
    temporary = path.with_suffix(path.suffix + ".partial")
    torch.save(data, temporary)
    temporary.replace(path)


def state_on_cpu(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def save_grid(samples, path, columns=8, title=None):
    samples = samples.detach().cpu().reshape(-1, 28, 28).numpy()
    rows = math.ceil(len(samples) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 1.05, rows * 1.05), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, sample in zip(axes.flat, samples):
        ax.imshow(sample, cmap="gray", vmin=0, vmax=1)
    if title:
        fig.suptitle(title, fontsize=11)
    fig.tight_layout(pad=.15, rect=(0, 0, 1, .95 if title else 1))
    fig.savefig(path, dpi=140)
    plt.close(fig)


@torch.no_grad()
def generate(model, kind, n, device, seed):
    model.eval()
    random = torch.Generator(device=device).manual_seed(seed)
    if kind == "gan":
        return (model(torch.randn(n, 100, device=device, generator=random)) + 1) / 2
    return model.sample(torch.rand(n, 784, device=device, generator=random))


@torch.no_grad()
def nll(model, data, device):
    model.eval()
    total = 0.
    for (batch,) in DataLoader(TensorDataset(data), batch_size=512):
        batch = batch.to(device)
        total += F.binary_cross_entropy_with_logits(model(batch), batch, reduction="sum").item()
    return total / len(data)


def plot_history(directory, history, kind):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
    if kind == "gan":
        for key, label in [("g_loss", "Generator non-saturating loss"), ("d_loss", "Discriminator loss")]:
            axes[0].plot(epochs, [row[key] for row in history], label=label)
        for key, label in [("d_real", "D(real)"), ("d_fake", "D(fake)")]:
            axes[1].plot(epochs, [row[key] for row in history], label=label)
        axes[1].set(ylabel="Mean discriminator probability", ylim=(0, 1))
        axes[0].set(ylabel="Mean BCE objective (not likelihood)")
    else:
        axes[0].plot(epochs, [row["train_nll"] for row in history], label="Training (online batches)")
        axes[0].plot(epochs, [row["validation_nll"] for row in history], label="Held-out validation")
        axes[0].set(ylabel="NLL (nats/image)")
        axes[1].plot(epochs, [row["optimization_seconds"] for row in history], label="Optimization time")
        axes[1].set(ylabel="Seconds per epoch")
    for ax in axes:
        ax.set(xlabel="Epoch")
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.savefig(directory / "learning_curves.png", dpi=170)
    plt.close(fig)


def run_experiment(config, data, output, device, smoke=False):
    directory = output / config.name
    directory.mkdir(parents=True, exist_ok=True)
    configuration = {**asdict(config), "smoke_test": smoke, "split_seed": 5494,
                     "binary_threshold": .5, "schema_version": 1}
    config_path = directory / "config.json"
    if config_path.exists():
        if json.loads(config_path.read_text()) != configuration:
            raise ValueError(f"Config differs from saved run {directory}; choose a new output directory")
    else:
        atomic_json(config_path, configuration)
    summary_path = directory / "summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary.get("complete"):
            print(config.name, "already complete; retained without retraining", flush=True)
            return summary
    torch.manual_seed(config.seed)
    np.random.seed(config.seed)
    is_gan = config.model == "gan"
    train = preprocess(data["train"][:config.n_train], binary=not is_gan)
    validation = preprocess(data["validation"][:256] if smoke else data["validation"], binary=True)
    test = preprocess(data["test"][:256] if smoke else data["test"], binary=True)
    if is_gan:
        model, discriminator = Generator().to(device), Discriminator().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, betas=(.5, .999))
        d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=config.lr, betas=(.5, .999))
    else:
        model = (FVSBN() if config.model == "fvsbn" else MADE(hidden=config.hidden, depth=config.depth)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    history, snapshots, best_state = [], [], None
    best_validation, best_epoch = float("inf"), 0
    first_epoch, previous_wall = 1, 0.
    checkpoint_path = directory / "last.pt"
    if checkpoint_path.exists():
        # Only read checkpoints generated in this run directory by this script.
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        if is_gan:
            discriminator.load_state_dict(checkpoint["discriminator"])
            d_optimizer.load_state_dict(checkpoint["d_optimizer"])
        history, snapshots = checkpoint["history"], checkpoint["snapshots"]
        best_state, best_validation, best_epoch = checkpoint["best_state"], checkpoint["best_validation"], checkpoint["best_epoch"]
        first_epoch = checkpoint["epoch"] + 1
        previous_wall = checkpoint["wall_seconds"]
        torch.set_rng_state(checkpoint["torch_rng"])
        if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
            torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
        print(config.name, "resuming at epoch", first_epoch, flush=True)
    started = time.perf_counter()
    for epoch in range(first_epoch, config.epochs + 1):
        loader_generator = torch.Generator().manual_seed(config.seed + epoch)
        loader = DataLoader(TensorDataset(train), batch_size=config.batch_size, shuffle=True,
                            generator=loader_generator, num_workers=0, pin_memory=device.type == "cuda")
        model.train()
        total, g_total, d_total, real_total, fake_total = 0., 0., 0., 0., 0.
        synchronize(device)
        start = time.perf_counter()
        for (batch,) in loader:
            batch = batch.to(device, non_blocking=True)
            n = len(batch)
            if is_gan:
                discriminator.train()
                for p in discriminator.parameters():
                    p.requires_grad_(True)
                d_optimizer.zero_grad(set_to_none=True)
                noise = torch.randn(n, 100, device=device)
                fake = model(noise).detach()
                real_logits, fake_logits = discriminator(batch), discriminator(fake)
                d_loss = F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
                d_loss += F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))
                d_loss.backward()
                d_optimizer.step()
                # Freeze D parameters while propagating its input gradient into G.
                for p in discriminator.parameters():
                    p.requires_grad_(False)
                optimizer.zero_grad(set_to_none=True)
                generated = model(torch.randn(n, 100, device=device))
                logits = discriminator(generated)
                g_loss = F.binary_cross_entropy_with_logits(logits, torch.ones_like(logits))
                g_loss.backward()
                optimizer.step()
                g_total += g_loss.item() * n
                d_total += d_loss.item() * n
                real_total += real_logits.sigmoid().detach().sum().item()
                fake_total += fake_logits.sigmoid().detach().sum().item()
            else:
                optimizer.zero_grad(set_to_none=True)
                loss = F.binary_cross_entropy_with_logits(model(batch), batch, reduction="none").sum(1).mean()
                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite autoregressive training loss")
                loss.backward()
                optimizer.step()
                total += loss.item() * n
        synchronize(device)
        seconds = time.perf_counter() - start
        row = {"epoch": epoch, "optimization_seconds": seconds}
        if is_gan:
            row.update(g_loss=g_total/len(train), d_loss=d_total/len(train),
                       d_real=real_total/len(train), d_fake=fake_total/len(train))
        else:
            validation_score = nll(model, validation, device)
            row.update(train_nll=total/len(train), validation_nll=validation_score)
            if validation_score < best_validation:
                best_validation, best_epoch, best_state = validation_score, epoch, state_on_cpu(model)
        history.append(row)
        # Same random values at every stage: changes reflect the model, not the seed.
        snapshots.append(generate(model, config.model, 4, device, config.seed + 1000).cpu())
        checkpoint = {"epoch": epoch, "model": state_on_cpu(model), "optimizer": optimizer.state_dict(),
                      "history": history, "snapshots": snapshots, "best_state": best_state,
                      "best_validation": best_validation, "best_epoch": best_epoch,
                      "torch_rng": torch.get_rng_state(),
                      "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
                      "wall_seconds": previous_wall + time.perf_counter() - started}
        if is_gan:
            checkpoint.update(discriminator=state_on_cpu(discriminator), d_optimizer=d_optimizer.state_dict())
        atomic_checkpoint(checkpoint_path, checkpoint)
        atomic_json(directory / "history.json", history)
        print(config.name, json.dumps(row), flush=True)
    if not is_gan:
        model.load_state_dict(best_state)
    model.eval()
    atomic_checkpoint(directory / "selected_model.pt", {"model": state_on_cpu(model), "config": configuration})
    # Final test likelihood is evaluated once after validation-based selection.
    test_nll = None if is_gan else nll(model, test, device)
    n_samples = 16 if smoke else 64
    _ = generate(model, config.model, 2, device, config.seed + 3000)  # inference warmup
    inference = []
    for repeat in range(3):
        synchronize(device)
        start = time.perf_counter()
        samples = generate(model, config.model, n_samples, device, config.seed + 2000)
        synchronize(device)
        inference.append(time.perf_counter() - start)
    np.save(directory / "samples.npy", samples.cpu().numpy())
    save_grid(samples, directory / "samples.png", title=config.name + (" (SMOKE TEST)" if smoke else ""))
    stages = np.unique(np.linspace(0, len(snapshots) - 1, min(5, len(snapshots))).round().astype(int))
    progress = torch.cat([snapshots[i] for i in stages])
    save_grid(progress, directory / "generation_progress.png", columns=4,
              title="Fixed randomness; epochs " + ", ".join(str(int(i)+1) for i in stages))
    plot_history(directory, history, config.model)
    summary = {"complete": True, "smoke_test": smoke, "config": configuration,
               "selection": "final fixed epoch" if is_gan else "minimum validation NLL",
               "selected_epoch": config.epochs if is_gan else best_epoch,
               "parameters_allocated": sum(p.numel() for p in model.parameters()),
               "parameters_active": sum(int(m.mask.sum()) + m.out_features for m in model.modules() if hasattr(m, "mask")) if not is_gan else sum(p.numel() for p in model.parameters()),
               "n_validation": len(validation) if not is_gan else None, "n_test": len(test) if not is_gan else None,
               "validation_nll_nats": None if is_gan else best_validation, "test_nll_nats": test_nll,
               "test_bits_per_dimension": None if is_gan else test_nll / (784 * math.log(2)),
               "training_optimization_seconds": sum(row["optimization_seconds"] for row in history),
               "run_wall_seconds": previous_wall + time.perf_counter() - started,
               "inference_samples_per_repeat": n_samples, "inference_seconds": inference,
               "inference_seconds_median": float(np.median(inference)),
               "sample_seed": config.seed + 2000, "progress_epochs": [int(i)+1 for i in stages],
               "device": str(device), "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
               "python": platform.python_version(), "torch": torch.__version__,
               "sample_sha256": hashlib.sha256((directory / "samples.npy").read_bytes()).hexdigest()}
    atomic_json(summary_path, summary)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--suite", choices=["main", "ablations", "all"], default="main")
    parser.add_argument("--only", help="One experiment name; restricts the selected suite")
    parser.add_argument("--device", choices=["auto", "cpu", "cuda"], default="auto")
    parser.add_argument("--data", type=Path, default=Path("data"))
    parser.add_argument("--output", type=Path, default=Path("runs"))
    parser.add_argument("--smoke", action="store_true", help="512 training images, 2 epochs: correctness only")
    args = parser.parse_args()
    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU detected. Select a GPU runtime in Colab, or pass --device cpu.")
    device = torch.device("cuda" if args.device == "auto" and torch.cuda.is_available() else "cpu" if args.device == "auto" else args.device)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    data = load_data(args.data)
    configurations = copy.deepcopy(MAIN if args.suite == "main" else ABLATIONS if args.suite == "ablations" else MAIN + ABLATIONS)
    if args.only:
        configurations = [c for c in configurations if c.name == args.only]
        if not configurations:
            raise ValueError("--only is not in the selected suite")
    if args.smoke:
        for c in configurations:
            c.epochs, c.n_train, c.hidden = 2, 512, 64
    args.output.mkdir(parents=True, exist_ok=True)
    summaries = [run_experiment(c, data, args.output, device, args.smoke) for c in configurations]
    atomic_json(args.output / ("suite_" + args.suite + ("_" + args.only if args.only else "") + ".json"), summaries)
    print("Finished", len(summaries), "experiments. Results:", args.output.resolve(), flush=True)


if __name__ == "__main__":
    main()


### diagnostics.py

In [ ]:
%%writefile diagnostics.py
"""Auxiliary diversity evidence, not a substitute for inspecting generated images.

Fit a small classifier on the training split only, assess it on validation, and
report predicted classes/confidence of the unfiltered saved samples. Report
nearest training neighbours and pairwise distances to help inspect memorization.
"""
import argparse
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from data import load_data
from train import atomic_json, save_grid


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=Path("data"))
    parser.add_argument("--output", type=Path, default=Path("runs"))
    parser.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    torch.set_num_threads(2)
    torch.manual_seed(5495)
    device = torch.device(args.device)
    dataset = load_data(args.data)
    n = 1000 if args.smoke else 50000
    train = dataset["train"][:n].float() / 255
    labels = dataset["train_labels"][:n].long()
    classifier = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=.001)
    for epoch in range(1 if args.smoke else 5):
        for x, y in DataLoader(TensorDataset(train, labels), batch_size=256, shuffle=True,
                               generator=torch.Generator().manual_seed(5495 + epoch)):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(classifier(x), y)
            loss.backward()
            optimizer.step()
    classifier.eval()
    nv = 256 if args.smoke else 10000
    with torch.no_grad():
        validation = dataset["validation"][:nv].float().to(device) / 255
        y = dataset["validation_labels"][:nv].to(device)
        accuracy = (classifier(validation).argmax(1) == y).float().mean().item()
        records = {}
        for summary_path in sorted(args.output.glob("*/summary.json")):
            summary = json.loads(summary_path.read_text())
            if summary["smoke_test"] != args.smoke:
                raise ValueError("Cannot mix smoke and formal results")
            directory = summary_path.parent
            samples = torch.from_numpy(np.load(directory / "samples.npy")).to(device)
            probabilities = classifier(samples).softmax(1)
            confidence, prediction = probabilities.max(1)
            counts = torch.bincount(prediction, minlength=10).cpu().tolist()
            # Use matching binary/grayscale preprocessing for nearest-neighbour analysis.
            reference = train[:summary["config"]["n_train"]]
            reference = (reference >= .5).float() if summary["config"]["model"] != "gan" else reference
            best_distance = torch.full((len(samples),), float("inf"), device=device)
            best_index = torch.zeros(len(samples), dtype=torch.long, device=device)
            for start in range(0, len(reference), 2048):
                distances = torch.cdist(samples, reference[start:start+2048].to(device)) / np.sqrt(784)
                value, index = distances.min(1)
                improved = value < best_distance
                best_distance[improved] = value[improved]
                best_index[improved] = index[improved] + start
            pairs = torch.stack([samples[:8].cpu(), reference[best_index[:8].cpu()]], dim=1).flatten(0, 1)
            save_grid(pairs, directory / "nearest_training_pairs.png", columns=4,
                      title="Generated / nearest training image (alternating)")
            pairwise = torch.pdist(samples) / np.sqrt(784)
            record = {"n_unfiltered_samples": len(samples), "classifier_predicted_counts": counts,
                      "classifier_mean_confidence": confidence.mean().item(),
                      "predicted_classes_present": sum(c > 0 for c in counts),
                      "fraction_confidence_above_0_8": (confidence >= .8).float().mean().item(),
                      "mean_pairwise_pixel_rmse": pairwise.mean().item(),
                      "nearest_training_mean_pixel_rmse": best_distance.mean().item(),
                      "nearest_training_indices": best_index.cpu().tolist(),
                      "nearest_training_pool_size": len(reference),
                      "limitation": "Classifier predictions may be wrong on generated images. Pixel distances are not perceptual quality or a proof against memorization. Inspect sample grids."}
            atomic_json(directory / "diagnostics.json", record)
            records[directory.name] = record
    atomic_json(args.output / "diagnostics.json", {"classifier_validation_accuracy": accuracy,
                "classifier_training_count": n, "smoke_test": args.smoke, "experiments": records})
    print("Auxiliary classifier validation accuracy:", accuracy, flush=True)


if __name__ == "__main__":
    main()


### export_results.py

In [ ]:
%%writefile export_results.py
"""Create a review packet with real result tables and figures, and export a ZIP."""
import argparse
import csv
import hashlib
import json
from pathlib import Path
import zipfile


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", type=Path, default=Path("runs"))
    parser.add_argument("--include-checkpoints", action="store_true")
    args = parser.parse_args()
    results = []
    for path in sorted(args.output.glob("*/summary.json")):
        result = json.loads(path.read_text())
        results.append(result)
    if not results:
        raise RuntimeError("No completed experiment results to export")
    rows = [{"experiment": r["config"]["name"], "smoke_test": r["smoke_test"],
             "n_train": r["config"]["n_train"], "epochs": r["config"]["epochs"],
             "lr": r["config"]["lr"], "selected_epoch": r["selected_epoch"],
             "test_nll_nats": r["test_nll_nats"], "test_bits_per_dimension": r["test_bits_per_dimension"],
             "training_seconds": r["training_optimization_seconds"],
             "inference_seconds": r["inference_seconds_median"],
             "n_generated_per_timing": r["inference_samples_per_repeat"], "device": r["device"]} for r in results]
    with (args.output / "comparison.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    lines = ["# MNIST experiment review packet", "", "Zian LI — student ID 60444432 (EID zli934). Final report filename: 60444432.pdf.", "",
             "This packet contains measured outputs. It is not the final assignment report.", ""]
    if any(r["smoke_test"] for r in results):
        lines += ["**SMOKE TEST ONLY: reduced data/epochs. Do not present these as trained-model results.**", ""]
    for r in results:
        name = r["config"]["name"]
        lines += ["## " + name, "", f"Configuration: `{json.dumps(r['config'])}`.", "",
                  f"Selected epoch: {r['selected_epoch']} ({r['selection']}). Training optimization time: {r['training_optimization_seconds']:.2f} s.", ""]
        if r["test_nll_nats"] is not None:
            lines += [f"Test NLL: {r['test_nll_nats']:.3f} nats/image; {r['test_bits_per_dimension']:.5f} bits/dimension.", ""]
        lines += [f"![Loss curves]({name}/learning_curves.png)", "", f"![Fixed-randomness progress]({name}/generation_progress.png)", "",
                  f"![Unfiltered samples]({name}/samples.png)", ""]
    lines += ["## Interpretation checklist", "",
              "- Compare FVSBN and MADE likelihood only on the same fixed binary test set. GAN has no reported tractable likelihood.",
              "- Compare depth and data ablations against MADE main; the data-size experiment fixes epochs, so fewer samples also mean fewer updates.",
              "- Compare GAN learning-rate curves, fixed-noise progress and unfiltered samples. Loss alone cannot establish visual quality or absence of mode collapse.",
              "- Report actual GPU model, training time, sample count and inference time. AR and GAN have different sampling computation.",
              "- Inspect any classifier diagnostics with their validation accuracy and generated-image distribution shift in mind.",
              "- Use author Zian LI, student ID 60444432, filename 60444432.pdf, and a frozen accessible code/data link in the final report."]
    (args.output / "REVIEW_RESULTS.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    files = sorted(p for p in args.output.rglob("*") if p.is_file() and p.suffix in
                   ({".json", ".csv", ".png", ".npy", ".md", ".pt"} if args.include_checkpoints else {".json", ".csv", ".png", ".npy", ".md"}))
    manifest = {p.relative_to(args.output).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
    archive = args.output.parent / (args.output.name + "_results.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for path in files:
            z.write(path, path.relative_to(args.output))
        z.writestr("export_manifest.json", json.dumps(manifest, indent=2))
    print("Exported", archive.resolve(), "bytes", archive.stat().st_size)


if __name__ == "__main__":
    main()


## 3. Mathematical verification

In [ ]:
subprocess.run([sys.executable, "check_models.py"], check=True)


## 4. Train the three required models
All models use the fixed official training split. FVSBN and MADE use binary pixels;
GAN uses original grayscale. The first epoch log provides an actual throughput
measurement. Checkpoints are saved after every epoch. Rerunning resumes them.


In [ ]:
common = ["--device", DEVICE, "--data", str(DATA), "--output", str(OUTPUT)]
if SMOKE:
    common.append("--smoke")
subprocess.run([sys.executable, "train.py", "--suite", "main", *common], check=True)


## 5. Controlled ablations
MADE: hidden depth 1 versus 2; 50k versus 10k training samples at a fixed epoch
budget. GAN: learning rate 2e-4 versus 1e-4. Keep this output directory/configuration
unchanged when resuming. These comparisons are needed for the experiment analysis.


In [ ]:
subprocess.run([sys.executable, "train.py", "--suite", "ablations", *common], check=True)


## 6. Auxiliary diversity and nearest-neighbour diagnostics
The auxiliary classifier uses training labels only and reports validation accuracy.
Its generated-image predictions are imperfect evidence, not ground-truth labels.
Always inspect the unfiltered sample grids as well.


In [ ]:
subprocess.run([sys.executable, "diagnostics.py", *common], check=True)


## 7. Inspect real outputs

In [ ]:
import json
from IPython.display import display, Image
for path in sorted(OUTPUT.glob("*/summary.json")):
    result = json.loads(path.read_text())
    print(path.parent.name, "selected epoch", result["selected_epoch"],
          "test NLL", result["test_nll_nats"], "train seconds", result["training_optimization_seconds"])
    display(Image(filename=str(path.parent / "samples.png")))
    display(Image(filename=str(path.parent / "generation_progress.png")))
    display(Image(filename=str(path.parent / "learning_curves.png")))


## 8. Export for report preparation
Download this ZIP and save the executed notebook. Checkpoints remain in Drive.
Set `INCLUDE_CHECKPOINTS=True` for a larger archive containing trained models too.
No results or quality claims are fabricated when a run is incomplete.


In [ ]:
INCLUDE_CHECKPOINTS = False
command = [sys.executable, "export_results.py", "--output", str(OUTPUT)]
if INCLUDE_CHECKPOINTS:
    command.append("--include-checkpoints")
subprocess.run(command, check=True)
archive = OUTPUT.parent / (OUTPUT.name + "_results.zip")
print("Download:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))


## Report completion
Use `comparison.csv`, `REVIEW_RESULTS.md`, histories, generated images and diagnostic
figures in the final report. Include the numeric student ID and an accessible,
frozen code/data link. Do not present smoke metrics as formal experiments or
compare GAN BCE to autoregressive NLL. Discuss observed failures and seed limits.

References: [MADE](https://proceedings.mlr.press/v37/germain15.html),
[GAN](https://arxiv.org/abs/1406.2661),
[original MNIST mirror/checksums](https://github.com/pytorch/vision/blob/main/torchvision/datasets/mnist.py).
